# Econ 5200 PS4 Warm-up — the code PS4 asks you to write

**Not graded.** Practice for Problem Set 4, on different data. Each exercise
shows you working code (*Watch*), asks you to write a similar piece yourself
(*Your turn*), then checks it. The exercises follow PS4's order: a world with the truth on file, covariate adjustment and how strong a hidden confounder would have to be, IV with its first stage first, a fuzzy RD with its checks, then one interface tested where each design should work.

- **Turn AI code completion off** (Colab: *Settings › AI assistance*), as PS4
  Phases 1–3 require. You may still click **Explain error**.
- The *Watch* cells use homes that can get an insulation grant and the *Your turn* cells use farms that can get a drip-irrigation subsidy. The two
  differ **on purpose**: you write each piece yourself instead of copying it.
- Run the cells in order: later exercises use what earlier ones made.
- If a check fails, read its message, fix your cell, and run both again.
- Stuck for more than five minutes? Look back at the *Watch* cell above it.
- Phase 4's specification curve is AI work and is not practised here.

In [ ]:
# Colab does not have these three: linearmodels for 2SLS (as in Lab 12), the RD packages as in Lab 13
!pip install linearmodels rdrobust==2.0.0 rddensity==3.0 -q

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS
from rdrobust import rdrobust
from rddensity import rddensity

---
## 1. A world with the truth on file, and a naive gap with the wrong sign (PS4 Phase 1)

Write the world in causal order: the hidden trait first, then what you observe of it, then the instrument and the running variable, each drawn **independently** of the hidden trait (that is what lets you use them later), then take-up, then the outcome. Two habits make the rest of PS4 checkable. Keep each unit's own effect in a column, `effect`: any average of it (over everyone, the treated, the compliers, the units at the cutoff) is then a truth you can compute. And keep `pull`, the wish to take up **before** the instrument acts: later it tells you whom the instrument moved. A cutoff that decides eligibility means an ineligible unit cannot take up, so multiply by `eligible`. The hidden trait raises both take-up and the outcome, which pushes the naive gap **up**; the gap still comes out negative because the grant goes to the homes that were colder to begin with.

**Watch:** run this and read it.

In [ ]:
rng = np.random.default_rng(12)
n_homes = 10_000
upkeep = rng.normal(0, 1, n_homes)                        # hidden: how well the owner keeps the house
home_age = 50 - 8 * upkeep + rng.normal(0, 15, n_homes)   # observed, and only partly tracks upkeep
owner = (0.8 * upkeep + rng.normal(0, 1, n_homes) > 0).astype(int)
rating = rng.uniform(0, 100, n_homes)                     # the audit's energy rating: high = a tight house
eligible = (rating < 40).astype(int)                      # the grant is for leaky homes, BELOW the cutoff
visit = (rng.random(n_homes) < 0.5).astype(int)           # an installer's visit, streets drawn by lottery
pull = -0.3 + 0.8 * upkeep + rng.normal(0, 1, n_homes)    # the wish to insulate before any visit
insulated = eligible * (pull + 1.5 * visit > 0)           # the visit adds 1.5; ineligible homes cannot take it
effect = 2.0 - 0.02 * rating + 0.4 * upkeep               # each home's own gain, in degrees C
temp = (15 + 1.0 * upkeep + 0.1 * rating - 0.01 * home_age
        + effect * insulated + rng.normal(0, 1, n_homes))  # winter indoor temperature
homes = pd.DataFrame({"upkeep": upkeep, "home_age": home_age, "owner": owner, "rating": rating,
                      "eligible": eligible, "visit": visit, "pull": pull, "insulated": insulated,
                      "effect": effect, "temp": temp})

naive_homes = (homes.loc[homes["insulated"] == 1, "temp"].mean()
               - homes.loc[homes["insulated"] == 0, "temp"].mean())
true_ate_homes = homes["effect"].mean()                                    # over every home
true_att_homes = homes.loc[homes["insulated"] == 1, "effect"].mean()       # over the insulated ones
print(f"naive gap {naive_homes:+.3f} | true effect: every home {true_ate_homes:+.3f}, "
      f"insulated homes {true_att_homes:+.3f}")

**Your turn:** build `farms`, 10,000 farms with the columns `skill`, `years`, `soil`, `hectares`, `eligible`, `demo`, `pull`, `drip`, `effect`, `tonnes`. `skill` is Normal(0, 1) and hidden; `years` (farming) is 20 + 6·skill + Normal(0, 8); `soil` is 50 + 5·skill + Normal(0, 10); `hectares` is Uniform(1, 9); a farm is eligible **below 4 hectares**; `demo` is 1 for about half the farms, whose village was drawn for a demonstration visit; `pull` is −0.5 + 0.9·skill + Normal(0, 1); `drip` is eligible × (pull + 1.2·demo > 0); `effect` is 1.2 − 0.15·hectares + 0.3·skill (tonnes per hectare); `tonnes` is 2 + 0.5·skill + 0.6·hectares + 0.01·soil + effect·drip + Normal(0, 0.5). Then store the naive gap (drip farms minus the others) as `naive_gap` and the true average effect over every farm as `true_ate`.

In [ ]:
rng = np.random.default_rng(31)
n_farms = 10_000
# YOUR CODE HERE
farms = naive_gap = true_ate = None

In [ ]:
cols = ["skill", "years", "soil", "hectares", "eligible", "demo", "pull", "drip", "effect", "tonnes"]
assert farms is not None and len(farms) == 10_000, "farms: one row per farm, 10,000 rows"
assert set(cols) <= set(farms.columns), "farms needs the columns " + ", ".join(cols)
f = farms
assert f["hectares"].between(1, 9).all() and abs(f["hectares"].mean() - 5) < 0.15, "hectares: Uniform(1, 9)"
assert (f["eligible"] == (f["hectares"] < 4).astype(int)).all(), "eligible is 1 below 4 hectares, 0 from 4 up"
assert set(f["demo"]) == {0, 1} and abs(f["demo"].mean() - 0.5) < 0.03, "demo: 0/1, 1 for about half the farms"
assert abs(np.corrcoef(f["demo"], f["skill"])[0, 1]) < 0.05 and abs(np.corrcoef(f["hectares"], f["skill"])[0, 1]) < 0.05, \
    "draw demo and hectares independently of skill: that is what makes them usable later"
slope = lambda col: np.polyfit(f["skill"], f[col], 1)[0]   # how far col moves per unit of skill
assert abs(slope("years") - 6) < 0.5 and abs(slope("soil") - 5) < 0.6, "years = 20 + 6 skill + noise; soil = 50 + 5 skill + noise"
v = f["pull"] - (-0.5 + 0.9 * f["skill"])
assert 0.95 < v.std() < 1.05 and abs(v.mean()) < 0.05, "pull = -0.5 + 0.9 skill + Normal(0, 1)"
assert (f["drip"] == f["eligible"] * (f["pull"] + 1.2 * f["demo"] > 0)).all(), \
    "drip = eligible x (pull + 1.2 demo > 0): an ineligible farm never takes it"
assert np.allclose(f["effect"], 1.2 - 0.15 * f["hectares"] + 0.3 * f["skill"]), "effect = 1.2 - 0.15 hectares + 0.3 skill"
e = f["tonnes"] - (2 + 0.5 * f["skill"] + 0.6 * f["hectares"] + 0.01 * f["soil"] + f["effect"] * f["drip"])
assert 0.47 < e.std() < 0.53 and abs(e.mean()) < 0.03, "tonnes = 2 + 0.5 skill + 0.6 hectares + 0.01 soil + effect x drip + Normal(0, 0.5)"
gap = f.loc[f["drip"] == 1, "tonnes"].mean() - f.loc[f["drip"] == 0, "tonnes"].mean()
assert naive_gap is not None and abs(naive_gap - gap) < 1e-9, "naive_gap: mean tonnes on drip farms minus on the others"
assert true_ate is not None and abs(true_ate - f["effect"].mean()) < 1e-9, "true_ate: the mean of effect over every farm"
assert naive_gap < 0 < true_ate, "the naive gap should come out negative while the true effect is positive"
print(f"naive gap {naive_gap:+.3f} | true average effect {true_ate:+.3f}")
print("Skill pushes the gap up. Which part of the world pulls it below zero?")

---
## 2. Adjust for what you observe, then ask how strong the hidden part must be (PS4 Step 2.1)

Selection on observables regresses the outcome on the treatment and every observed **pre-treatment** covariate, the running variable included, since it drives both eligibility and the outcome. It is valid if, given those covariates, take-up is as good as random. Topic 10's Cinelli and Hazlett (2020) measure how exposed the estimate is on the partial R² scale. A regressor's partial R² is the share of the leftover variation it explains, t² / (t² + df). From the treatment's t and the residual df, f = t / √df and the **robustness value** RV = ½(√(f⁴ + 4f²) − f²): the partial R² a confounder needs with **both** the treatment and the outcome to bring the estimate to zero. Benchmark RV against an observed covariate: its partial R² with the treatment (from the regression of the treatment on the covariates) and with the outcome (from the adjusted regression). The bias formula then says how far a confounder as strong as that covariate would move the estimate. Only a simulation can also fit the long regression with the hidden trait in it: the formula reproduces that gap to every digit.

**Watch:** run this and read it.

In [ ]:
covs_homes = "home_age + owner + rating"
adj_homes = smf.ols(f"temp ~ insulated + {covs_homes}", data=homes).fit()
print(f"adjusted {adj_homes.params['insulated']:.3f} (se {adj_homes.bse['insulated']:.3f}) | "
      f"truth on the insulated {true_att_homes:.3f}, on every home {true_ate_homes:.3f}")


def partial_r2(fit, name):
    """The share of the leftover variation that one regressor explains: t^2 / (t^2 + df)."""
    t = fit.tvalues[name]
    return t**2 / (t**2 + fit.df_resid)


def bias_from(fit, treatment, r2_y, r2_d):
    """Cinelli-Hazlett: how far a confounder with these two partial R2 values moves the estimate."""
    return fit.bse[treatment] * np.sqrt(r2_y * r2_d / (1 - r2_d) * fit.df_resid)


f_homes = adj_homes.tvalues["insulated"] / np.sqrt(adj_homes.df_resid)
rv_homes = 0.5 * (np.sqrt(f_homes**4 + 4 * f_homes**2) - f_homes**2)

# The benchmark: owner, an observed covariate that tracks upkeep
take_homes = smf.ols(f"insulated ~ {covs_homes}", data=homes).fit()        # the treatment on the covariates
r2_d_owner = partial_r2(take_homes, "owner")
r2_y_owner = partial_r2(adj_homes, "owner")
print(f"robustness value {rv_homes:.3f} | owner: partial R2 {r2_d_owner:.4f} with insulated, "
      f"{r2_y_owner:.4f} with temp")
like_owner = bias_from(adj_homes, "insulated", r2_y_owner, r2_d_owner)
print(f"RV is {rv_homes / max(r2_d_owner, r2_y_owner):.0f} times owner's larger partial R2; a confounder as "
      f"strong as owner on both would move the estimate by {like_owner:.3f}")

# Only in a simulation: put upkeep in, and the formula with upkeep's own partial R2 gives the gap exactly
long_homes = smf.ols(f"temp ~ insulated + {covs_homes} + upkeep", data=homes).fit()
r2_y_u = partial_r2(long_homes, "upkeep")
r2_d_u = partial_r2(smf.ols(f"insulated ~ {covs_homes} + upkeep", data=homes).fit(), "upkeep")
print(f"without upkeep minus with it {adj_homes.params['insulated'] - long_homes.params['insulated']:.4f} | "
      f"formula {bias_from(adj_homes, 'insulated', r2_y_u, r2_d_u):.4f}   (upkeep: {r2_d_u:.3f} and {r2_y_u:.3f})")

**Your turn:** fit `adj_fit`, tonnes on drip, years, soil and hectares. Compute its robustness value as `rv`. Benchmark against `years`: `r2_d_years` (its partial R² with drip, given soil and hectares), `r2_y_years` (with tonnes, given drip and the rest) and `bias_like_years`, how far a confounder as strong as years on both would move the estimate. Use the two functions from the *Watch* cell.

In [ ]:
# YOUR CODE HERE
adj_fit = rv = r2_d_years = r2_y_years = bias_like_years = None

In [ ]:
assert adj_fit is not None and {"drip", "years", "soil", "hectares"} <= set(adj_fit.params.index), \
    "adj_fit: tonnes ~ drip + years + soil + hectares"
assert len(adj_fit.params) == 5, "adj_fit: the intercept, drip and the three covariates, nothing else"
ref = smf.ols("tonnes ~ drip + years + soil + hectares", data=farms).fit()
ref_take = smf.ols("drip ~ years + soil + hectares", data=farms).fit()
assert abs(adj_fit.params["drip"] - ref.params["drip"]) < 1e-9, "adj_fit should be fitted on farms"
fr = ref.tvalues["drip"] / np.sqrt(ref.df_resid)
assert rv is not None and abs(rv - 0.5 * (np.sqrt(fr**4 + 4 * fr**2) - fr**2)) < 1e-9, \
    "rv = 0.5 (sqrt(f^4 + 4 f^2) - f^2), with f = t / sqrt(df) for drip"
assert r2_d_years is not None and abs(r2_d_years - partial_r2(ref_take, "years")) < 1e-9, \
    "r2_d_years: from drip ~ years + soil + hectares, years' t^2 / (t^2 + df)"
assert r2_y_years is not None and abs(r2_y_years - partial_r2(ref, "years")) < 1e-9, \
    "r2_y_years: from the adjusted regression of tonnes, years' t^2 / (t^2 + df)"
assert bias_like_years is not None and abs(bias_like_years - bias_from(ref, "drip", r2_y_years, r2_d_years)) < 1e-9, \
    "bias_like_years: bias_from(adj_fit, 'drip', r2_y_years, r2_d_years)"
true_att = farms.loc[farms["drip"] == 1, "effect"].mean()
assert adj_fit.params["drip"] > true_att, "the hidden skill should push the adjusted estimate above the truth"
print(f"adjusted {adj_fit.params['drip']:.3f} | truth on the drip farms {true_att:.3f}, on every farm {true_ate:.3f}")
print(f"RV {rv:.3f} = {rv / max(r2_d_years, r2_y_years):.0f} times years' larger partial R2 "
      f"({r2_d_years:.4f}, {r2_y_years:.4f}); a confounder that strong moves it by {bias_like_years:.3f}")
print("Which assumption makes the adjusted estimate valid? Write the sentence a referee could argue with.")

---
## 3. The first stage before anything else, then whose effect 2SLS estimates (PS4 Step 2.2)

Report the first stage and its F **first**. With one instrument the F is the instrument's t squared. It measures relevance only: it says nothing about exclusion. Topic 12's number is **104.7**: Lee, McCrary, Moreira and Porter (2022) show a 5% t-test on 2SLS keeps its size only above that F, so the old F > 10 rule is not the standard. Then 2SLS: `IV2SLS.from_formula("y ~ 1 + controls + [d ~ z]", data=df).fit()` is linearmodels' formula form of Lab 12's `IV2SLS`, with robust standard errors. The controls are not needed for identification here (the visit was drawn by lottery); they soak up variation and shrink the standard error. 2SLS estimates a LATE: the average effect for the **compliers**, the units the instrument moved. In a simulation you can name them: eligible, `pull` too low to take up alone, and high enough with the instrument's push.

**Watch:** run this and read it.

In [ ]:
first_homes = smf.ols("insulated ~ visit + home_age + owner + rating", data=homes).fit()
F_homes = first_homes.tvalues["visit"] ** 2                 # one instrument: F is its t squared
print(f"first stage: the visit raises take-up by {first_homes.params['visit']:.3f}, F = {F_homes:.0f} "
      f"({'above' if F_homes > 104.7 else 'below'} 104.7)")

iv_homes = IV2SLS.from_formula("temp ~ 1 + home_age + owner + rating + [insulated ~ visit]", data=homes).fit()

# Who did the visit move? Eligible, would not insulate alone, would with the visit's push of 1.5
complier_homes = (homes["eligible"] == 1) & (homes["pull"] <= 0) & (homes["pull"] + 1.5 > 0)
late_homes = homes.loc[complier_homes, "effect"].mean()
print(f"compliers: {complier_homes.mean():.3f} of homes (the first stage estimates this share)")
print(f"2SLS {iv_homes.params['insulated']:.3f} (se {iv_homes.std_errors['insulated']:.3f}) | true effect: "
      f"compliers {late_homes:.3f}, insulated homes {true_att_homes:.3f}, every home {true_ate_homes:.3f}")

# The second stage run by hand gives the same estimate, but its OLS standard error is wrong
homes["ins_hat"] = first_homes.fittedvalues
by_hand = smf.ols("temp ~ ins_hat + home_age + owner + rating", data=homes).fit()
classical = IV2SLS.from_formula("temp ~ 1 + home_age + owner + rating + [insulated ~ visit]",
                                data=homes).fit(cov_type="unadjusted")
print(f"by hand {by_hand.params['ins_hat']:.3f}, se {by_hand.bse['ins_hat']:.3f} | "
      f"2SLS's classical se {classical.std_errors['insulated']:.3f}")
print("Too large here; on the Topic 12 deck's Card slide it was too small. Report the package's.")

**Your turn:** on `farms`, with years, soil and hectares as controls in both stages: fit the first stage as `first_fit` and store its F for `demo` as `first_F`; fit 2SLS as `iv_fit`. Then, from the simulation, store the share of farms that are compliers as `complier_share` and their mean effect as `late_true`.

In [ ]:
# YOUR CODE HERE
first_fit = first_F = iv_fit = complier_share = late_true = None

In [ ]:
assert first_fit is not None and {"demo", "years", "soil", "hectares"} <= set(first_fit.params.index), \
    "first_fit: drip ~ demo + years + soil + hectares"
ref_first = smf.ols("drip ~ demo + years + soil + hectares", data=farms).fit()
assert first_F is not None and abs(first_F - ref_first.tvalues["demo"] ** 2) < 1e-6, \
    "first_F: the square of demo's t-statistic in the first stage, with the controls"
assert iv_fit is not None and "drip" in iv_fit.params.index, "iv_fit: IV2SLS with [drip ~ demo]"
ref_iv = IV2SLS.from_formula("tonnes ~ 1 + years + soil + hectares + [drip ~ demo]", data=farms).fit()
assert abs(iv_fit.params["drip"] - ref_iv.params["drip"]) < 1e-9, "iv_fit: the same three controls as the first stage"
want = (farms["eligible"] == 1) & (farms["pull"] <= 0) & (farms["pull"] + 1.2 > 0)
assert complier_share is not None and abs(complier_share - want.mean()) < 1e-12, \
    "complier_share: eligible, pull <= 0, and pull + 1.2 > 0"
assert late_true is not None and abs(late_true - farms.loc[want, "effect"].mean()) < 1e-12, \
    "late_true: mean effect over the compliers"
assert abs(first_fit.params["demo"] - complier_share) < 0.03, "the first stage should land near the complier share"
print(f"first stage {first_fit.params['demo']:.3f} (compliers {complier_share:.3f}), F = {first_F:.0f}: relevance only")
print(f"2SLS {iv_fit.params['drip']:.3f} (se {iv_fit.std_errors['drip']:.3f}) | compliers {late_true:.3f}, "
      f"every farm {true_ate:.3f}")
print("Which farms are the compliers, in words? Give one argument for the exclusion restriction and one against.")

---
## 4. A fuzzy RD whose treated side is below the cutoff (PS4 Step 2.3)

`rdrobust` reports right minus left, and the course puts the treated side on the right. When eligibility is **below** the cutoff, recentre: `dist = cutoff - running` is positive for the eligible units, and every jump then reads as the effect of being eligible. Eligibility here changes the *chance* of treatment, so the design is **fuzzy**: `rdrobust(y=..., x=..., c=0, fuzzy=d)` divides the jump in the outcome by the jump in take-up, and the estimand is the effect for compliers **at the cutoff**. Report the robust row's interval and p-value beside the estimate. The bandwidth is a bias–variance choice, so report the estimate at half and double the chosen h as well. Then the two checks: `rddensity` for bunching of the running variable, and an RD with a pre-treatment covariate as the outcome. Report both, even when they pass. One draw near a cutoff is noisy: the *Watch* cell ends with the same fuzzy RD averaged over 100 new worlds.

**Watch:** run this and read it.

In [ ]:
homes["dist"] = 40 - homes["rating"]              # rating points below 40: eligible homes on the right
take_jump_homes = rdrobust(y=homes["insulated"], x=homes["dist"], c=0)
backwards = rdrobust(y=homes["insulated"], x=homes["rating"], c=40)    # not recentred, for comparison
print(f"jump in take-up: {take_jump_homes.coef.loc['Conventional', 'Coeff']:+.3f} recentred, "
      f"{backwards.coef.loc['Conventional', 'Coeff']:+.3f} on the raw rating")

rd_homes = rdrobust(y=homes["temp"], x=homes["dist"], c=0, fuzzy=homes["insulated"])
h_homes = rd_homes.bws.loc["h", "left"]
rows = []
for h in [h_homes / 2, h_homes, 2 * h_homes]:                  # narrower: less bias; wider: less noise
    fit = rd_homes if h == h_homes else rdrobust(y=homes["temp"], x=homes["dist"], c=0,
                                                 fuzzy=homes["insulated"], h=h)
    rows.append({"h": h, "estimate": fit.coef.loc["Conventional", "Coeff"],
                 "robust_low": fit.ci.loc["Robust", "CI Lower"], "robust_high": fit.ci.loc["Robust", "CI Upper"],
                 "robust_p": fit.pv.loc["Robust", "P>|z|"]})
p_fmt = {"robust_p": lambda p: f"{p:.3f}" if p >= 0.001 else "< 0.001"}   # so a tiny p is not printed as 0
print(pd.DataFrame(rows).round(3).to_string(index=False, formatters=p_fmt))

density_homes = rddensity(X=homes["dist"].to_numpy(), c=0).test["p_jk"]
age_jump = rdrobust(y=homes["home_age"], x=homes["dist"], c=0)
print(f"density test p {density_homes:.3f} | jump in home_age {age_jump.coef.loc['Conventional', 'Coeff']:+.2f} "
      f"years, robust p {age_jump.pv.loc['Robust', 'P>|z|']:.3f}")

# The truth at the cutoff: homes within 2 points of 40 that would insulate if eligible
near = homes["dist"].abs() < 2
takes_if_eligible = homes["pull"] + 1.5 * homes["visit"] > 0
print(f"true effect for compliers at the cutoff {homes.loc[near & takes_if_eligible, 'effect'].mean():.3f}")

# The other end: the same fuzzy RD on 100 new worlds, drawn as homes was (home_age and owner left out:
# they do not jump at the cutoff), each with its own truth at the cutoff
draws, truths = [], []
for s in range(100):
    r = np.random.default_rng(500 + s)
    u, rt, v = r.normal(0, 1, n_homes), r.uniform(0, 100, n_homes), (r.random(n_homes) < 0.5).astype(int)
    takes = -0.3 + 0.8 * u + r.normal(0, 1, n_homes) + 1.5 * v > 0     # would insulate if eligible
    d = (rt < 40) * takes
    eff = 2.0 - 0.02 * rt + 0.4 * u
    y = 15 + 1.0 * u + 0.1 * rt + eff * d + r.normal(0, 1, n_homes)
    draws.append(rdrobust(y=y, x=40 - rt, c=0, fuzzy=d).coef.loc["Conventional", "Coeff"])
    truths.append(eff[(np.abs(40 - rt) < 2) & takes].mean())
print(f"100 new worlds: mean estimate {np.mean(draws):.3f}, spread (sd) {np.std(draws, ddof=1):.3f} | "
      f"mean truth at the cutoff {np.mean(truths):.3f}")

**Your turn:** on `farms`, eligible below 4 hectares: make `farms["dist"]` so the eligible farms are on the right. Store the jump in take-up as `take_jump` and the fuzzy RD of tonnes as `rd_fit`. Build `bw_table`, a DataFrame with the columns `h`, `estimate`, `robust_low`, `robust_high`, `robust_p`, one row each at half, once and twice `rd_fit`'s bandwidth. Store the density test's p-value as `density_p` and the robust p-value of an RD with `soil` as the outcome as `soil_p`. Last, `cutoff_truth`: the mean effect of the farms within 0.2 hectares of 4 that would take drip if eligible.

In [ ]:
# YOUR CODE HERE
take_jump = rd_fit = bw_table = density_p = soil_p = cutoff_truth = None

In [ ]:
assert "dist" in farms and np.allclose(farms["dist"], 4 - farms["hectares"]), \
    "dist = 4 - hectares, so the eligible farms (below 4) have dist > 0, on the right"
x = farms["dist"]
ref_take = rdrobust(y=farms["drip"], x=x, c=0)
ref_rd = rdrobust(y=farms["tonnes"], x=x, c=0, fuzzy=farms["drip"])
coef = lambda fit: float(fit.coef.loc["Conventional", "Coeff"])
assert take_jump is not None and abs(coef(take_jump) - coef(ref_take)) < 1e-9, "take_jump: rdrobust of drip on dist, c=0"
assert coef(take_jump) > 0, "take-up should jump UP to the right of the cutoff once dist is recentred"
assert rd_fit is not None and abs(coef(rd_fit) - coef(ref_rd)) < 1e-9, \
    "rd_fit: rdrobust of tonnes on dist, c=0, fuzzy=farms['drip']"
assert bw_table is not None and list(bw_table.columns)[:5] == ["h", "estimate", "robust_low", "robust_high", "robust_p"], \
    "bw_table columns: h, estimate, robust_low, robust_high, robust_p"
h0 = ref_rd.bws.loc["h", "left"]
assert len(bw_table) == 3 and np.allclose(bw_table["h"], [h0 / 2, h0, 2 * h0]), "three rows: h/2, h, 2h of rd_fit's h"
for _, row in bw_table.iterrows():
    fit = rdrobust(y=farms["tonnes"], x=x, c=0, fuzzy=farms["drip"], h=row["h"])
    assert abs(row["estimate"] - coef(fit)) < 1e-9 and abs(row["robust_p"] - fit.pv.loc["Robust", "P>|z|"]) < 1e-9, \
        f"bw_table at h = {row['h']:.3f}: the fuzzy RD's conventional estimate and its robust p-value"
    assert row["robust_low"] < row["robust_high"], "robust_low and robust_high: the robust row's interval"
assert density_p is not None and abs(density_p - rddensity(X=x.to_numpy(), c=0).test["p_jk"]) < 1e-9, \
    "density_p: rddensity on dist at 0, .test['p_jk']"
assert soil_p is not None and abs(soil_p - rdrobust(y=farms["soil"], x=x, c=0).pv.loc["Robust", "P>|z|"]) < 1e-9, \
    "soil_p: the robust p-value of rdrobust with soil as the outcome"
want = farms.loc[(x.abs() < 0.2) & (farms["pull"] + 1.2 * farms["demo"] > 0), "effect"].mean()
assert cutoff_truth is not None and abs(cutoff_truth - want) < 1e-12, \
    "cutoff_truth: farms within 0.2 of the cutoff whose pull + 1.2 demo > 0"
print(bw_table.round(3).to_string(index=False, formatters=p_fmt))
print(f"density p {density_p:.3f} | soil robust p {soil_p:.3f} | true effect for compliers at the cutoff {cutoff_truth:.3f}")
print(f"Three truths now: compliers to the visit {late_true:.3f}, compliers at the cutoff {cutoff_truth:.3f}, "
      f"every farm {true_ate:.3f}. Which population does each design speak for?")

---
## 5. One interface, tested where the design should work (PS4 Step 3.3)

Give every estimator the same shape: it takes the data frame and column names, and returns a dictionary with the same keys, here `estimate`, `se` and `n`. Then one test fits them all: is the estimate within four standard errors of the truth? Test each design on a world where **its own** assumption holds by construction, over several seeds, so a pass is not one lucky draw. And show the test can fail: run the same estimator on a world where its assumption is broken. A test that cannot fail tells you nothing. In PS4 these go into your module with docstrings and type hints; write them here as plain functions first.

**Watch:** run this and read it.

In [ ]:
def adjust_effect(df, outcome, treatment, covariates):
    """Selection on observables: OLS of the outcome on the treatment and the covariates."""
    fit = smf.ols(f"{outcome} ~ " + " + ".join([treatment] + list(covariates)), data=df).fit()
    return {"estimate": fit.params[treatment], "se": fit.bse[treatment], "n": int(fit.nobs)}


def recovers(result, truth, k=4):
    """True when the estimate is within k standard errors of the truth."""
    return abs(result["estimate"] - truth) <= k * result["se"]


def world_observed(seed, n=5_000, effect=1.2):
    """Take-up depends only on what is observed, so adjustment should work here."""
    r = np.random.default_rng(seed)
    age, rating = r.normal(50, 15, n), r.uniform(0, 100, n)
    took = (0.03 * (age - 50) - 0.02 * (rating - 50) + r.normal(0, 1, n) > 0).astype(int)
    temp = 15 + 0.1 * rating - 0.01 * age + effect * took + r.normal(0, 1, n)
    return pd.DataFrame({"age": age, "rating": rating, "took": took, "temp": temp})


hits = [recovers(adjust_effect(world_observed(s), "temp", "took", ["age", "rating"]), 1.2) for s in range(20)]
print(f"adjustment where its assumption holds: recovers 1.2 on {sum(hits)} of 20 seeds")
# The known negative: homes, where upkeep is hidden, so the same test should fail
on_homes = adjust_effect(homes, "temp", "insulated", ["home_age", "owner", "rating"])
print(f"adjustment on homes: {on_homes['estimate']:.3f} against {true_att_homes:.3f}, "
      f"recovers: {recovers(on_homes, true_att_homes)}")

**Your turn:** write `iv_effect(df, outcome, treatment, instrument, covariates)`, 2SLS with linearmodels, returning the same three keys as `adjust_effect` (`n` as a whole number). Write `world_iv(seed, n=5_000, effect=0.7)`: a hidden `skill` from Normal(0, 1), a 0/1 `demo` drawn for half the farms, `drip` = (−0.2 + 0.9·skill + 1.0·demo + Normal(0, 1) > 0) as 0/1, and `tonnes` = 3 + 0.5·skill + effect·drip + Normal(0, 0.5); return the columns `skill`, `demo`, `drip`, `tonnes` (skill is kept for checking only). Then store `iv_hits`, the list of `recovers(...)` results of `iv_effect` with no covariates on seeds 0 to 19, and `ols_there`, `recovers(...)` for `adjust_effect` with no covariates on `world_iv(0)`.

In [ ]:
# YOUR CODE HERE
iv_effect = world_iv = iv_hits = ols_there = None

In [ ]:
assert callable(iv_effect) and callable(world_iv), "define iv_effect and world_iv as functions"
w = world_iv(3)
assert len(w) == 5_000 and {"skill", "demo", "drip", "tonnes"} <= set(w.columns), "world_iv: 5,000 rows, skill, demo, drip, tonnes"
assert not w.equals(world_iv(4)) and w.equals(world_iv(3)), "world_iv(seed) should start its own generator from seed"
assert set(w["demo"]) == {0, 1} and set(w["drip"]) == {0, 1}, "demo and drip are 0/1"
assert abs(np.corrcoef(w["demo"], w["skill"])[0, 1]) < 0.06, "demo is drawn independently of skill"
e = w["tonnes"] - (3 + 0.5 * w["skill"] + 0.7 * w["drip"])
assert 0.46 < e.std() < 0.54, "tonnes = 3 + 0.5 skill + effect x drip + Normal(0, 0.5)"
got = iv_effect(w, "tonnes", "drip", "demo", [])
assert isinstance(got, dict) and set(got) == {"estimate", "se", "n"}, "iv_effect returns a dict with estimate, se, n"
ref = IV2SLS.from_formula("tonnes ~ 1 + [drip ~ demo]", data=w).fit()
assert abs(got["estimate"] - ref.params["drip"]) < 1e-9 and abs(got["se"] - ref.std_errors["drip"]) < 1e-9, \
    "iv_effect: IV2SLS with [treatment ~ instrument], its params and std_errors"
assert got["n"] == 5_000 and isinstance(got["n"], int), "n: the number of rows used, as an int"
with_cov = iv_effect(farms, "tonnes", "drip", "demo", ["years", "soil", "hectares"])
assert abs(with_cov["estimate"] - iv_fit.params["drip"]) < 1e-9, "iv_effect should put the covariates in both stages, as iv_fit did"
assert iv_hits is not None and len(iv_hits) == 20 and all(iv_hits), "iv_hits: 20 results, every one True"
assert ols_there is not None and not ols_there, "ols_there: OLS on world_iv(0) should fail the test, since skill is hidden there"
print(f"2SLS where its assumptions hold: recovers 0.7 on {sum(iv_hits)} of 20 seeds")
print(f"OLS on the same world recovers it: {ols_there}. The test can fail, so its passes mean something.")

---
**Done.** A world with the truth on file and a naive gap of the wrong sign, an adjusted estimate with its robustness value benchmarked against an observed covariate, a first stage reported before its 2SLS and set against the complier truth, a fuzzy RD recentred so the treated side is on the right with its bandwidth table and two checks, and one interface tested where its design should work: the building blocks of PS4 Phases 1–3. PS4 asks for the RD estimator in that interface too. Now open `ps4-starter.ipynb`.